<a href="https://colab.research.google.com/github/jaw039/min-viable-eeg/blob/main/Run_Review_Repeated_Experimental_Seeds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Run/Review Repeated Experimental Seeds

In [ ]:
#Identify High-Variance Conditions

def identify_high_variance(results_df, threshold=0.05):
    """Find experimental conditions with high variance."""

    high_variance = []

    for (budget, method), group in results_df.groupby(['budget', 'method']):
        std_kappa = group['kappa'].std()
        mean_kappa = group['kappa'].mean()
        cv = std_kappa / mean_kappa  # Coefficient of variation

        if cv > threshold:
            high_variance.append({
                'budget': budget,
                'method': method,
                'mean_kappa': mean_kappa,
                'std_kappa': std_kappa,
                'cv': cv,
                'n_runs': len(group)
            })

    return pd.DataFrame(high_variance)

In [ ]:
#Re-run Critical Experiments

def repeat_experiments(high_variance_conditions, extra_seeds=10):
    """Re-run experiments with additional seeds."""

    additional_results = []

    for _, row in high_variance_conditions.iterrows():
        print(f"Re-running: Budget={row['budget']}, Method={row['method']}")

        for seed in range(extra_seeds):
            # Set seed
            np.random.seed(seed + 1000)  # Use different seed range

            # Re-run experiment
            result = run_single_experiment(
                budget=row['budget'],
                method=row['method'],
                seed=seed + 1000
            )
            additional_results.append(result)

    return pd.DataFrame(additional_results)

In [ ]:
#Verify Consistency

def verify_consistency(original_results, repeated_results):
    """Verify that repeated results are consistent with original."""

    consistency_check = []

    for (budget, method), group_orig in original_results.groupby(['budget', 'method']):
        group_rep = repeated_results[
            (repeated_results['budget'] == budget) &
            (repeated_results['method'] == method)
        ]

        if len(group_rep) == 0:
            continue

        # Compare means
        orig_mean = group_orig['kappa'].mean()
        rep_mean = group_rep['kappa'].mean()
        mean_diff = abs(orig_mean - rep_mean)

        # Statistical test
        from scipy import stats
        t_stat, p_value = stats.ttest_ind(group_orig['kappa'], group_rep['kappa'])

        consistency_check.append({
            'budget': budget,
            'method': method,
            'original_mean': orig_mean,
            'repeated_mean': rep_mean,
            'mean_difference': mean_diff,
            'p_value': p_value,
            'consistent': p_value > 0.05
        })

    return pd.DataFrame(consistency_check)